# Initialization

## Pseudocode: Greedy Knapsack GA Initialization
1. **Input:** item list `items`, capacity `C`, population size `P`.
2. Sort items by value/weight ratio in descending order.
3. Build a **base chromosome** by scanning sorted items and setting `x[j] = 1` if adding item `j` keeps total weight ≤ `C`.
4. Add `base` to the population.
5. **For** each remaining individual (until size `P`):
   - Copy `base` into `x`.
   - Pick one item currently selected in `x`, set it to 0 (free some capacity).
   - Scan unselected items in ratio order and turn on the first that fits the freed capacity.
   - Append the repaired chromosome `x` to the population.
6. **Return** the greedy-biased population.

In [ ]:
# Greedy initialization for Knapsack GA
from typing import List, Tuple

Item = Tuple[int, int]  # (value, weight)

def greedy_chromosome(items: List[Item], capacity: int) -> List[int]:
    """Pick items in descending value/weight ratio until the knapsack is full."""
    ratio_order = sorted(
        enumerate(items),
        key=lambda entry: entry[1][0] / entry[1][1],
        reverse=True,
    )
    chromosome = [0] * len(items)
    total_weight = 0
    for idx, (_, weight) in ratio_order:
        if total_weight + weight <= capacity:
            chromosome[idx] = 1
            total_weight += weight
    return chromosome

def initialize_population_greedy(items: List[Item], capacity: int, population_size: int) -> List[List[int]]:
    """Create a population where each chromosome is a greedy permutation with slight perturbations."""
    base = greedy_chromosome(items, capacity)
    population = [base]
    for _ in range(population_size - 1):
        perturbed = base.copy()
        # Try swapping one picked item with an unpicked item to diversify while staying greedy-ish
        picked = [i for i, gene in enumerate(perturbed) if gene]
        unpicked = [i for i, gene in enumerate(perturbed) if not gene]
        if not picked or not unpicked:
            population.append(perturbed)
            continue
        drop_idx = picked[0]
        perturbed[drop_idx] = 0
        total_weight = sum(items[i][1] for i, g in enumerate(perturbed) if g)
        for add_idx in unpicked:
            candidate_weight = items[add_idx][1]
            if total_weight + candidate_weight <= capacity:
                perturbed[add_idx] = 1
                total_weight += candidate_weight
                break
        population.append(perturbed)
    return population

# Demo
items = [(10, 5), (6, 4), (3, 2), (7, 3), (18, 9)]
capacity = 12
population = initialize_population_greedy(items, capacity, population_size=5)
population

[5, 4, 3]
[5, 2, 3, 9]
[5, 2, 9]
[9]
[3]
[4, 2, 9]


[[1, 1, 0, 1, 0],
 [0, 0, 0, 1, 1],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 1, 0],
 [0, 1, 1, 0, 0]]

# Selection Methods

## Roulette Wheel Selection
Given fitness scores $f_i$ for $N$ individuals, assign probabilities
$$p_i = \frac{f_i}{\sum_{j=1}^{N} f_j}$$
The selection procedure:
1. Normalize the fitness values to obtain the cumulative distribution.
2. Sample a random value $r \in [0,1)$.
3. Choose the first individual whose cumulative probability exceeds $r$.
4. Repeat until the desired number of parents is collected (individuals may be selected multiple times).

In [ ]:
import random
from typing import Callable, List, Sequence

Chromosome = List[int]

def roulette_wheel_selection(
    population: Sequence[Chromosome],
    fitness_fn: Callable[[Chromosome], float],
    parent_count: int,
) -> List[Chromosome]:
    scores = [max(fitness_fn(ind), 0.0) for ind in population]
    total = sum(scores)
    if total == 0:
        # fallback to uniform probability
        probabilities = [1 / len(population)] * len(population)
    else:
        probabilities = [score / total for score in scores]

    cumulative: List[float] = []
    running = 0.0
    for p in probabilities:
        running += p
        cumulative.append(running)
    cumulative[-1] = 1.0

    parents: List[Chromosome] = []
    for _ in range(parent_count):
        r = random.random()
        for idx, threshold in enumerate(cumulative):
            if r <= threshold:
                parents.append(population[idx])
                break
    return parents

# Demo
items_demo = [(10, 5), (6, 4), (3, 2), (7, 3), (18, 9)]
capacity_demo = 12

def knapsack_fitness(chromosome: Chromosome) -> float:
    total_value = 0
    total_weight = 0
    for gene, (value, weight) in zip(chromosome, items_demo):
        if gene:
            total_value += value
            total_weight += weight
    return total_value if total_weight <= capacity_demo else 0.0

pop = [[1,0,0,1,0], [1,1,0,0,0], [0,0,1,1,0], [1,1,1,0,0]]
selected = roulette_wheel_selection(pop, knapsack_fitness, parent_count=3)
selected

[0.2222222222222222, 0.2222222222222222, 0.2222222222222222, 0.3333333333333333]


[[1, 0, 0, 1], [1, 1, 1, 0], [1, 1, 1, 0]]

## Elitism (Survivor Selection)
- Ordene a população corrente por aptidão (melhor valor primeiro).
- Preserve os `elitism_size` indivíduos de topo sem modificações.
- Complete o restante da nova geração selecionando candidatos via torneio, roleta ou outro método, respeitando a quantidade `population_size - elitism_size`.
- Combine elites e candidatos escolhidos para formar a próxima população.
- Avalie novamente os indivíduos (se necessário) antes da próxima geração.

In [ ]:
from typing import Callable, List, Sequence, Tuple
import random

Chromosome = List[int]

def elitism_survivor_selection(
    population: Sequence[Chromosome],
    fitness_fn: Callable[[Chromosome], float],
    offspring: Sequence[Chromosome],
    elitism_size: int,
) -> List[Chromosome]:
    """Preserve top performers and refill with offspring ranked by fitness."""
    combined = list(population)
    scored: List[Tuple[float, Chromosome]] = [
        (fitness_fn(ind), ind) for ind in combined
    ]
    scored.sort(key=lambda pair: pair[0], reverse=True)
    elites = [ind for _, ind in scored[:elitism_size]]

    remaining_slots = max(0, len(population) - elitism_size)
    offspring_scored = sorted(
        ((fitness_fn(child), child) for child in offspring),
        key=lambda pair: pair[0],
        reverse=True,
    )
    survivors = [child for _, child in offspring_scored[:remaining_slots]]
    if len(survivors) < remaining_slots:
        survivors.extend(child for _, child in offspring_scored[remaining_slots:remaining_slots*2])
    return elites + survivors

# Demo
items_demo = [(10, 5), (6, 4), (3, 2), (7, 3), (18, 9)]
capacity_demo = 12

def knapsack_fitness(chromosome: Chromosome) -> float:
    total_value = 0
    total_weight = 0
    for gene, (value, weight) in zip(chromosome, items_demo):
        if gene:
            total_value += value
            total_weight += weight
    return total_value if total_weight <= capacity_demo else 0.0

parent_pop = [[1,0,1,0,0], [1,1,0,0,0], [0,0,1,1,0], [1,1,1,0,0]]
kids = [[1,0,0,1,0], [0,1,0,1,0], [1,1,1,1,0], [0,0,0,1,0]]
next_generation = elitism_survivor_selection(parent_pop, knapsack_fitness, kids, elitism_size=2)
next_generation